## Good context to coding agent must cover following 3 categories: 

1. **Code context**: The codebase structure, module responsibilities, core data structures, and coding standards. Without this information, the Agent may produce code that is syntactically correct but inconsistent with the project’s style or architecture.
2. **Process requirements**: Git branching strategy, commit conventions, review process, and CI/CD requirements. Without this information, the Agent may commit untested code directly to the main branch.
3. **Environment configuration**: Development setup, test database connection strings, test-environment deployment procedures, and API key management practices. Without this information, a fix that works locally may fail immediately in the test environment.

## 5 components of context:

1. System prompt
2. tool definitions
3. User message
4. Assistant message (contains nl responses + tool calls)
5. Tool call results

3-5 are collectively called 'trajectory' of conversation.

OpenAI format chat completions api (the format pretty much every model uses) stores these components as a *messages* list. Messages can have one of 4 roles - 

1. System: system prompt, usually just a single big prompt at the beginning of the list
2. User: user query
3. Assistant: nl responses + actual tool calls objects
4. Tool: tool call results. Linked to corresponding assistant message containing tool call via `tool_call_id`

*tool definitions* aren't stored in *messages* here, they're sent separately in a *tools* field to the model.

1. Tool Definitions example (maintained by dev)
    ```python
    [
        {
            "type": "function",
            "function": {
                "name": "get_current_time",
                "description": "Get the current date and time in a specific timezone",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "timezone": {
                            "type": "string",
                            "description": "Timezone name, e.g. America/Vancouver",
                        }
                    },
                },
            },
        },
        {
            "type": "function",
            "function": {
                "name": "get_weather",
                "description": "Get the current weather for a specific city",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "city": {"type": "string", "description": "City name"},
                        "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                    },
                },
            },
        },
    ]

    ```


2. Messages example (maintained via code)

    ```python
    [
        {
            "role": "system",
            "content": "You are a helpful assistant. Use the provided tools to get real-time information when needed.",
        },
        {"role": "user", "content": "What's the current time and weather in Vancouver?"},
    ]
    ```

3. Reply by model api / completions api (could be tool call or response)

    ```python
    {
        "choices": [
            {
                "message": {
                    "role": "assistant",
                    "content": None,
                    "tool_calls": [
                        {
                            "id": "call_abc123",
                            "type": "function",
                            "function": {
                                "name": "get_current_time",
                                "arguments": '{"timezone": "America/Vancouver"}',
                            },
                        },
                        {
                            "id": "call_def456",
                            "type": "function",
                            "function": {
                                "name": "get_weather",
                                "arguments": '{"city": "Vancouver", "unit": "celsius"}',
                            },
                        },
                    ],
                }
            }
        ]
    }
    ```
    These tool calls are also added verbatim, as assistant messages.
    
    PS: model can match the tool result vs tool call by checking id.

## Code

In [4]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

client = OpenAI()

# ── Tool definitions ──
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_time",
            "description": "Get the current date and time in a specific timezone",
            "parameters": {
                "type": "object",
                "properties": {
                    "timezone": {"type": "string", "description": "Timezone name, e.g. America/Vancouver"}
                },
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a specific city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "City name"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
            },
        },
    },
]

# ── Tool execution function (stub with canned results; a real implementation
#    must parse the JSON `arguments` and call actual APIs) ──
def execute_tool(name, arguments):
    if name == "get_current_time":
        return '{"datetime": "2025-09-13T05:18:47", "day_of_week": "Saturday"}'
    elif name == "get_weather":
        return '{"temperature": 13.2, "unit": "celsius", "conditions": "clear", "humidity": 93}'

# ── Initial message list ──
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use tools to get real-time information when needed."},
    {"role": "user", "content": "What's the current time and weather in Vancouver?"},
]

# ── Agent core loop ──
# Production code needs a max_iterations cap here: as discussed later in
# this chapter, Agents can become stuck repeating the same tool calls forever
while True:
    response = client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, tools=tools
    )
    assistant_message = response.choices[0].message

    # Append model's response to message list (whether text or tool calls)
    messages.append(assistant_message)

    # If no tool calls requested, the model has produced its final response
    if not assistant_message.tool_calls:
        print(assistant_message.content)
        break

    # Execute each tool requested by the model, append results to message list
    for tool_call in assistant_message.tool_calls:
        result = execute_tool(tool_call.function.name, tool_call.function.arguments)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": result,
        })

    print("full messages list")
    print(messages)
    # Return to top of loop, call model again with updated message list

full messages list
[{'role': 'system', 'content': 'You are a helpful assistant. Use tools to get real-time information when needed.'}, {'role': 'user', 'content': "What's the current time and weather in Vancouver?"}, ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_AZxZmD3Y7ol46LjCaunX45j3', function=Function(arguments='{"timezone": "America/Vancouver"}', name='get_current_time'), type='function'), ChatCompletionMessageFunctionToolCall(id='call_wOGzVl88KjDvxHC80KQtxn02', function=Function(arguments='{"city": "Vancouver", "unit": "celsius"}', name='get_weather'), type='function')]), {'role': 'tool', 'tool_call_id': 'call_AZxZmD3Y7ol46LjCaunX45j3', 'content': '{"datetime": "2025-09-13T05:18:47", "day_of_week": "Saturday"}'}, {'role': 'tool', 'tool_call_id': 'call_wOGzVl88KjDvxHC80KQtxn02', 'content': '{"temperature": 13.2, "unit": "celsius", "conditions": "clear",